## 0. Import libraries

In [ ]:
import json
from datasets import * 
from pathlib import Path

## 1. Dataset Preprocess

In [ ]:
def convert_to_json(file_path: str, json_path):
    current_sub = ""
    current_sentences = ""
    current_dict = []
    current_label = None

    json_data = []

    print("Extracting data...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: # Empty line -> last sentence has finished
                if current_label is not None and current_sub != "":
                    current_dict.append(
                        {
                            "entity_text": current_sub,
                            "entity_label": current_label
                        }
                    )   
                if current_sentences:          
                    json_data.append(
                        {
                            "sentence": current_sentences,
                            "entity_info": current_dict
                        }
                    )        
                current_sub = ""
                current_sentences = ""
                current_dict = []
                current_label = None                

            else:
                chunk = line.split(" ")
                if len(chunk) == 2:
                    current_sentences += chunk[0]
                    if chunk[1] == current_label:
                        current_sub += chunk[0]
                    elif chunk[1].startswith('B'):
                        if current_label is not None and current_sub != "":
                            current_dict.append(
                                {
                                    "entity_text": current_sub,
                                    "entity_label": current_label
                                }
                            )

                        # Assign new label 
                        label = chunk[1].split('-') # e.g. B-中药
                        current_label = label[1] if len(label) == 2 else None
                        current_sub = chunk[0]
                    else: # chunk[1] == 'O'
                        if current_label is not None and current_sub != "":
                            current_dict.append(
                                {
                                    "entity_text": current_sub,
                                    "entity_label": current_label
                                }
                            )
                        current_label = None
                        current_sub = ""
    print(f"Converting {len(json_data)} sentences with their entity information to a jsonal file.")
    with open(json_path, 'a', encoding='utf-8') as f:
        for obj in json_data:
            f.write(json.dumps(obj) + "\n")
    print("Convert Successfully.")
                    

train_dataset_path = "./data/medical.train"
train_json_dataset_path = "./data/medical_train.jsonl"
train_json_dataset_path = Path(train_json_dataset_path)
train_json_dataset_path.touch()
convert_to_json(train_dataset_path, train_json_dataset_path)
test_data = "./data/medical.test"
test_json_dataset_path = "./data/medical_test.jsonl"
test_json_dataset_path = Path(test_json_dataset_path)
test_json_dataset_path.touch()
convert_to_json(test_data, test_json_dataset_path)
val_data = "./data/medical.val"
val_json_dataset_path = "./data/medical_val.jsonl"
val_json_dataset_path = Path(val_json_dataset_path)
val_json_dataset_path.touch()
convert_to_json(val_data, val_json_dataset_path)

train_dataset = Dataset.from_json(train_json_dataset_path)
test_dataset = Dataset.from_json(test_json_dataset_path)
val_dataset = Dataset.from_json(val_json_dataset_path)